In [3]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [9]:
df = pd.read_csv("dataset.csv", encoding='utf-8', engine='python', on_bad_lines='skip')

In [11]:
print(df.shape)
df.head()
df.columns

(27703, 2)


Index(['review', 'sentiment'], dtype='object')

In [12]:
print(df.shape)
print(df['sentiment'].value_counts())

(27703, 2)
sentiment
positive    13880
negative    13823
Name: count, dtype: int64


In [13]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [14]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]

    # Lemmatization (or use stemming)
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [15]:
df['clean_text'] = df['review'].apply(preprocess_text)

In [16]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

In [17]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

In [18]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

In [19]:
y = df['sentiment']

In [20]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [21]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

In [22]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

In [23]:
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)
y_pred_dt = dt.predict(X_test_bow)

In [24]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"----- {model_name} -----")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [25]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")

----- Logistic Regression -----
Accuracy: 0.8900920411478073
Precision: 0.890192747667433
Recall: 0.8900920411478073
F1 Score: 0.8900727386817441

Classification Report:
               precision    recall  f1-score   support

    negative       0.90      0.88      0.89      2740
    positive       0.88      0.90      0.89      2801

    accuracy                           0.89      5541
   macro avg       0.89      0.89      0.89      5541
weighted avg       0.89      0.89      0.89      5541

----- Naive Bayes -----
Accuracy: 0.8494856524093124
Precision: 0.849650730951806
Recall: 0.8494856524093124
F1 Score: 0.8494875054889894

Classification Report:
               precision    recall  f1-score   support

    negative       0.84      0.86      0.85      2740
    positive       0.86      0.84      0.85      2801

    accuracy                           0.85      5541
   macro avg       0.85      0.85      0.85      5541
weighted avg       0.85      0.85      0.85      5541

----- Decisi